# bridge

> IPython namespace bridge

One indirection with three jobs: pick the namespace (IPython `user_ns` by default,
or whatever `set_namespace` pointed at for headless `serve()`), expose the frontend-agnostic
verbs (`snapshot`/`view`/`expand`/`grid`/`set_value`) over it, and hook `post_run_cell` so
in-kernel frontends refresh after every execution. All server routes and the agent overlay
resolve the live namespace through this module, so switching what paar inspects is a single
call.

In [ ]:
#| default_exp bridge

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import paar.bridge as B
from paar.bridge import Bridge, on_change

class _Events:
    def __init__(self): self.cbs=[]
    def register(self, name, fn): self.cbs.append((name, fn))
class _IP:
    def __init__(self, ns): self.user_ns=ns; self.user_ns_hidden={'_ih'}; self.events=_Events()

def test_bridge_snapshot_uses_ns_and_hidden():
    B.get_ipython = lambda: _IP({'a':1, '_ih':2})
    assert [r.name for r in Bridge().snapshot()]==['a']
def test_on_change_registers_and_fires():
    ip = _IP({}); B.get_ipython = lambda: ip
    hits=[]; ok = on_change(lambda: hits.append(1))
    assert ok is True and ip.events.cbs[0][0]=='post_run_cell'
    ip.events.cbs[0][1](None); assert hits==[1]
def test_no_ipython_is_safe():
    B.get_ipython = lambda: None
    assert Bridge().snapshot()==[] and on_change(lambda: None) is False
test_bridge_snapshot_uses_ns_and_hidden(); test_on_change_registers_and_fires(); test_no_ipython_is_safe()

In [ ]:
def test_bridge_expand():
    B.get_ipython = lambda: _IP({'d': {'x': 1, 'y': 2}})
    kids = Bridge().expand(('d',))
    assert [k.name for k in kids]==["'x'", "'y'"] and kids[0].value=='1'
test_bridge_expand()

def test_bridge_snapshot_filters():
    B.get_ipython = lambda: _IP({'alpha':1, 'beta':'x'})
    assert [r.name for r in Bridge().snapshot(name='a')] == ['alpha', 'beta']
    assert [r.name for r in Bridge().snapshot(typ='int')] == ['alpha']
def test_bridge_set_value():
    ip = _IP({'x': 1}); B.get_ipython = lambda: ip
    import paar.runtime as R; R.get_ipython = lambda: ip
    assert Bridge().set_value(('x',), '7') is None and ip.user_ns['x'] == 7
test_bridge_snapshot_filters(); test_bridge_set_value()

In [ ]:
def test_bridge_grid():
    import numpy as np
    B.get_ipython = lambda: _IP({'arr': np.arange(6).reshape(2,3)})
    p = Bridge().grid(('arr',))
    assert p['nrows']==2 and p['ncols']==3
test_bridge_grid()

In [ ]:
def test_bridge_view_groups():
    import math
    B.get_ipython = lambda: _IP({'myvar': 5, 'mymod': math})
    view = Bridge().view('standard')
    labels = [l for l,_ in view]
    assert None in labels and 'Modules' in labels
    assert [v.name for v in dict(view)[None]]==['myvar']
test_bridge_view_groups()

In [ ]:
def test_set_namespace_override():
    B.get_ipython = lambda: None            # no IPython at all
    B.set_namespace({'z': 42})
    try:
        assert [r.name for r in Bridge().snapshot()]==['z']   # reads the explicit ns
    finally:
        B.set_namespace(None)               # restore IPython path
    assert Bridge().snapshot()==[]          # override cleared
test_set_namespace_override()

In [ ]:
#| export
from paar.snapshot import snapshot, expand, grid_page, profile_view
from paar.runtime import set_value
try: from IPython import get_ipython
except Exception:
    def get_ipython(): return None

_ns_override = None   # set by paar.fasthtml.serve() to inspect a plain process's namespace

def set_namespace(ns):
    "Point the bridge at an explicit namespace dict (headless/non-IPython use); None restores the IPython path."
    global _ns_override; _ns_override = ns

def _ns():
    if _ns_override is not None: return _ns_override
    ip = get_ipython(); return ip.user_ns if ip else {}

def _hidden():
    if _ns_override is not None: return frozenset()
    ip = get_ipython()
    return frozenset(getattr(ip, 'user_ns_hidden', ())) if ip else frozenset()

class Bridge:
    "Frontend-agnostic access to the live kernel namespace."
    def snapshot(self, name=None, typ=None): return snapshot(_ns(), _hidden(), name, typ)
    def view(self, profile='standard'): return profile_view(_ns(), _hidden(), profile)
    def expand(self, accessor, offset=0): return expand(_ns(), accessor, offset)
    def grid(self, accessor, roff=0, coff=0, rows=50, cols=50): return grid_page(_ns(), accessor, roff, coff, rows, cols)
    def set_value(self, accessor, expr): return set_value(accessor, expr)

def on_change(cb):
    "Register cb() to fire after each cell execution; returns False outside IPython."
    ip = get_ipython()
    if not ip: return False
    ip.events.register('post_run_cell', lambda res: cb())
    return True

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()